## Extracción de datos del IPC desde la API del INE

### Objetivo
Este notebook tiene como finalidad la extracción, exploración y estructuración de los datos del **Índice de Precios de Consumo (IPC)** publicados por el **Instituto Nacional de Estadística (INE)** a través de su API pública TEMPUS. 

A partir de la serie de datos IPCA (IPC Armonizado) con periodicidad mensual, se construye un **modelo dimensional** compuesto por:
- **Dimensiones**: tabla de tiempo (*tiempo*), tabla de territorios (*territorio*), tabla de sectores IPC (*sectores_ipc*) y tabla de tipos de medida (*tipo_medida*).
- **Tabla de hechos**: tabla central (*ipc*) que relaciona cada observación del índice con sus dimensiones asociadas.

### Metodología
1. **Conexión a la API del INE** — Llamada al endpoint `/DATOS_TABLA/76136` para obtener los últimos 60 periodos mensuales.
2. **Extracción de dimensiones** — Recorrido de la respuesta JSON para poblar las tablas dimensionales, garantizando unicidad mediante filtros de clave.
3. **Construcción de la tabla de hechos** — Cruce de las dimensiones a través de identificadores primarios y asignación del valor del IPC a cada combinación (territorio, sector, medida, periodo).
4. **Exportación** — Volcado de cada tabla a ficheros CSV en `../files/data_raw/` para su consumo en etapas posteriores del pipeline.

### Contexto del proyecto
Estos datos se integran en un análisis más amplio sobre la **resiliencia empresarial en España**, donde se combinarán con información de constitución y disolución de empresas para estudiar la correlación entre el entorno macroeconómico (inflación) y la actividad empresarial.

In [ ]:
# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

import pandas as pd
# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación del módulo de conexión a la API
from src.api import conection_api as conexion_api
from src.api.config import API_URLS


In [ ]:
url = API_URLS["ipc"]

In [ ]:
data = conexion_api.llamada_api(url)

In [ ]:
len(data)

In [ ]:
data

In [ ]:
data[70]['Nombre']    #Ruta al nombre

In [ ]:
data[0]['Data'][0]['Periodo']['Mes_inicio']     #Ruta al mes

In [ ]:
data[0]['Data'][0]['Anyo']       #Ruta al año

In [ ]:
data[0]['Data'][0]['Periodo']['Nombre_largo']       #Ruta al nombre del mes

In [ ]:
data[0]['Data'][0]['Valor']             #Ruta al IPC

In [ ]:
data[0]['Data']

In [ ]:
''' sep50 = '=' * 50
sep100 = '=' * 100

partes = []
for serie in data:
    for dato in serie['Data']:
        partes.append(
            f"{sep50}\n{serie['Nombre']}\n{sep100}\n"
            f"- Numero Mes: {dato['Periodo']['Mes_inicio']}\n{sep100}\n"
            f"- Numero Año: {dato['Anyo']}\n{sep100}\n"
            f"- Mes: {dato['Periodo']['Nombre_largo']}\n{sep100}\n"
            f"- IPC: {dato['Valor']}\n{sep100}\n"
            f"-Codigo fecha: {dato['CodigoPeriodo']}\n{sep50}\n"
        )

print("\n".join(partes))'''

In [ ]:
''' for serie in data:
    for dato in serie['Data']:
        print(f'{'=*' * 50}\n{serie['Nombre']}\n{'=' * 100}\n- Numero Mes: {dato['Periodo']['Mes_inicio']}\n{'=' * 100}\n- Numero Año: {dato['Anyo']}\n{'=' * 100}\n- Mes: {dato['Periodo']['Nombre_largo']}\n{'=' * 100}\n- IPC: {dato['Valor']}\n{'=' * 100}\n-Codigo fecha: {dato['CodigoPeriodo']}\n{'=' * 50}\n') 
        '''

In [ ]:
tiempo = {'id_tiempo': [], 'anio': [], 'mes': [], 'nombre_mes': [] }

for serie in data:
    for dato in serie['Data']:
        if dato['CodigoPeriodo'] not in tiempo['id_tiempo']:
            tiempo['id_tiempo'].append(dato['CodigoPeriodo'])
            tiempo['anio'].append(dato['Anyo'])
            tiempo['mes'].append(dato['Periodo']['Mes_inicio'])
            tiempo['nombre_mes'].append(dato['Periodo']['Nombre_largo'])


In [ ]:
tiempo = pd.DataFrame(tiempo)
tiempo.sample()

In [ ]:
tiempo.shape

In [ ]:
tiempo.to_csv('../files/data_raw/tiempo.csv', index=False)

In [ ]:
territorio = {'id_territorio': [], 'nombre_territorio': []}
id_ter = 1

for serie in data:
    nombre_completo = serie['Nombre']
    nombre_limpio = nombre_completo.split('.')[0].strip()

    if nombre_limpio not in territorio['nombre_territorio'] and 'nacional' not in nombre_limpio.lower():
        territorio['id_territorio'].append(id_ter)
        territorio['nombre_territorio'].append(nombre_limpio)

        id_ter += 1

In [ ]:
territorio = pd.DataFrame(territorio)
territorio.sample()

In [ ]:
territorio.shape

In [ ]:
territorio.to_csv('../files/data_raw/territorio.csv', index=False)

In [ ]:
sectores_ipc = {'id_sector': [], 'nombre_sector': []}
id_sec = 1

for serie in data:
    nombre_completo = serie['Nombre']
    sector_limpio = nombre_completo.split('.')[1].strip()

    if sector_limpio not in sectores_ipc['nombre_sector']:
        sectores_ipc['id_sector'].append(id_sec)
        sectores_ipc['nombre_sector'].append(sector_limpio)

        id_sec += 1        

In [ ]:
sectores_ipc = pd.DataFrame(sectores_ipc)

In [ ]:
sectores_ipc

In [ ]:
sectores_ipc.to_csv('../files/data_raw/sectores_ipc.csv', index=False)

In [ ]:
tipo_medida = {'id_medida': [], 'nombre_medida': []}
id_med = 1

for serie in data:
    nombre_completo = serie['Nombre']
    medida_limpio = nombre_completo.split('.')[2].strip()

    if medida_limpio not in tipo_medida['nombre_medida']:
        tipo_medida['id_medida'].append(id_med)
        tipo_medida['nombre_medida'].append(medida_limpio)

        id_med += 1     

In [ ]:
tipo_medida

In [ ]:
tipo_medida = pd.DataFrame(tipo_medida)

In [ ]:
tipo_medida

In [ ]:
tipo_medida.to_csv('../files/data_raw/tipo_medida.csv', index=False)

In [ ]:
#  Creamos diccionarios de búsqueda rápida (Mapeos)
# Esto sirve para que Python asocie al vuelo un nombre con su ID.

map_territorios = dict(zip(territorio['nombre_territorio'], territorio['id_territorio']))
map_sectores = dict(zip(sectores_ipc['nombre_sector'], sectores_ipc['id_sector']))
map_medidas = dict(zip(tipo_medida['nombre_medida'], tipo_medida['id_medida']))
 
ipc = {
    'id_tiempo': [],
    'id_territorio': [],
    'id_sector': [],
    'id_medida': [],
    'valor_ipc': []
}
 
# Extracción y Cruce de IDs
for serie in data:
    nombre_completo = serie['Nombre']
    partes = nombre_completo.split('.')
    if len(partes) >= 3:

        # Extraemos los nombres de texto limpio
        territorio_texto = partes[0].strip()
        sector_texto = partes[1].strip()
        medida_texto = partes[2].strip()

        # Corregimos si dice 'Nacional' para que coincida con nuestra dimensión
        if territorio_texto.lower() == 'nacional':
            territorio_texto = 'Total Nacional'

        # Miramos en nuestros mapas qué ID le toca a cada texto
        id_terr = map_territorios.get(territorio_texto)
        id_sec = map_sectores.get(sector_texto)
        id_med = map_medidas.get(medida_texto)

        # Si por algún motivo el INE da un texto que no guardamos antes, nos lo saltamos
        if id_terr is None or id_sec is None or id_med is None:
            continue

        # Bajamos al segundo nivel: los datos históricos de esta serie
        for dato in serie['Data']:
            valor = dato.get('Valor')
            id_time = dato.get('CodigoPeriodo')     # Sacamos el código del periodo para usarlo como ID de tiempo

            # Si tenemos todos los datos, appendeamos en la tabla
            if valor is not None and id_time is not None:
                ipc['id_tiempo'].append(str(id_time))
                ipc['id_territorio'].append(id_terr)
                ipc['id_sector'].append(id_sec)
                ipc['id_medida'].append(id_med)
                ipc['valor_ipc'].append(valor)
 
 

In [ ]:
print(ipc) # Print de control

In [ ]:
ipc = pd.DataFrame(ipc)

ipc.sample(5)

In [ ]:
ipc.to_csv('../files/data_raw/ipc.csv', index=False)